In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
import matplotlib.pyplot as plt
import xarray as xr
import yaml
import geopandas as gpd
import contextily as cx
from adjustText import adjust_text
import itertools


from neuralhydrology.nh_run import start_run, eval_run, finetune
from neuralhydrology.nh_run import continue_run
from neuralhydrology.utils.config import Config
from neuralhydrology.evaluation import get_tester, metrics

In [2]:
# -------- Paths ---------
CONFIG_PATH = Path("./basins_subset_test.yml")
RUNS_DIR = Path("runs")
ATTRIBUTES_FILE= Path('../extended_dataset/data/attributes/attributes_other.csv')

In [3]:
# precip_products = [
#     # 'total_precipitation_sum'
#     # "chirps_precipitation",
#     "mswep_precipitation"
# ]

In [4]:
seeds = [111, 222, 333, 444, 555, 666, 777, 888] #  

base_non_precip_inputs = [
    "temperature_2m_max",
    "temperature_2m_min",
    "surface_net_solar_radiation_mean",
]

with open(CONFIG_PATH, "r") as f:
    base_config = yaml.safe_load(f)

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()

# for r in range(1, len(precip_products) + 1):

precip_combos = [
    ("camels_precipitation",),
    ("total_precipitation_sum",),
    ("chirps_precipitation",),
    ("mswep_precipitation",),
    ("chirps_precipitation", "mswep_precipitation"),
]

# for precip_combo in itertools.combinations(precip_products, r):
for precip_combo in precip_combos:
    for seed in seeds:
        config = base_config.copy()

        config["dynamic_inputs"] = [*base_non_precip_inputs, *precip_combo]
        config["seed"] = seed

        precip_name = "_".join(precip_combo)
        config["experiment_name"] = (
            f"{precip_name}_seq_{config['seq_length']}"
            f"_{config['predict_last_n']}_epochs_{config['epochs']}"
            f"_hidden_{config['hidden_size']}"
            f"_dropout_{str(config['output_dropout']).replace('.', '')}"
            f"_fb_{config['initial_forget_bias']}"
            f"_seed{seed}"
        )

        temp_config_path = Path(f"temp_{precip_name}_seed{seed}.yml")
        with open(temp_config_path, "w") as f:
            yaml.dump(config, f)

        print(f"Running: {config['experiment_name']}")

        if use_gpu:
            start_run(config_file=temp_config_path)
        else:
            start_run(config_file=temp_config_path, gpu=-1)

        temp_config_path.unlink()  # clean up temp file after run

Running: camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111
2026-04-29 18:33:46,782: Logging to /home/azureuser/sky_workdir/extending_caravan/127_basins_runs/runs/camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2904_183346/output.log initialized.
2026-04-29 18:33:46,783: ### Folder structure created at /home/azureuser/sky_workdir/extending_caravan/127_basins_runs/runs/camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_2904_183346
2026-04-29 18:33:46,783: ### Run configurations for camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111
2026-04-29 18:33:46,783: batch_size: 256
2026-04-29 18:33:46,784: clip_gradient_norm: 1
2026-04-29 18:33:46,784: data_dir: ../extended_dataset/data
2026-04-29 18:33:46,785: dataset: generic
2026-04-29 18:33:46,785: device: cuda:0
2026-04-29 18:33:46,786: dynamic_inputs: ['temperature_2m_max', 'temperature_2m_min', 'surface_net_solar_radiation_mean', 'camels_

# Testing

In [9]:
from neuralhydrology.nh_run import start_run, eval_run, finetune
from pathlib import Path
import pandas as pd
import pickle
import numpy as np
import yaml

In [10]:
RUN_DIR = Path("./runs")

In [12]:
# Find all directories starting with 'precip'
precip_dirs = [d for d in RUN_DIR.iterdir() if d.is_dir() and d.name.startswith(("camels", "total_precipitation", "mswep"))]

print(f"Found {len(precip_dirs)} directories: {[d.name for d in precip_dirs]}")

Found 24 directories: ['total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_055003', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_3004_065746', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_3004_080530', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_3004_112853', 'camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed888_3004_022707', 'mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_3004_213843', 'mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_0105_042703', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_3004_102104', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed111_3004_033441', 'mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_0105_021033', 'total_precipitation_sum_seq_270_1_epochs_30_hidden_

In [13]:
for run_dir in precip_dirs:

    # Skip if test folder already exists
    if (run_dir / "test").exists():
        print(f"Skipping {run_dir.name} — test folder already exists")
        continue

    config_path = run_dir / "config.yml"
    
    if not config_path.exists():
        print(f"Skipping {run_dir.name} — no config.yml found")
        continue
    
    print(f"\nProcessing {run_dir.name}...")
    
    # Read the config
    with open(config_path, 'r') as f:
        content = f.read()
    
    # Only add if not already present
    if "test_start_date" not in content:
        with open(config_path, 'a') as f:
            f.write("\ntest_start_date: 01/10/2009")
        print(f"  Added test_start_date")
    else:
        print(f"  test_start_date already present, skipping")

    if "test_end_date" not in content:
        with open(config_path, 'a') as f:
            f.write("\ntest_end_date: 30/09/2019")
        print(f"  Added test_end_date")
    else:
        print(f"  test_end_date already present, skipping")

    # Run evaluation
    print(f"  Running eval_run...")
    eval_run(run_dir=run_dir, period="test")
    print(f"  Done!")

print("\nAll runs completed!")

Skipping total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_055003 — test folder already exists

Processing total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_3004_065746...
  test_start_date already present, skipping
  test_end_date already present, skipping
  Running eval_run...
2026-05-01 16:06:37,565: Using the model weights from runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_3004_065746/model_epoch030.pt


/home/azureuser/miniconda3/envs/neuralhydrology/lib/python3.10/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.mod

# Evaluation:   0%|          | 0/127 [00:00<?, ?it/s]

# Evaluation: 100%|██████████| 127/127 [02:04<00:00,  1.02it/s]
2026-05-01 16:08:42,473: Stored metrics at runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_3004_065746/test/model_epoch030/test_metrics.csv
2026-05-01 16:08:42,504: Stored results at runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed444_3004_065746/test/model_epoch030/test_results.p
  Done!

Processing total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_3004_080530...
  test_start_date already present, skipping
  test_end_date already present, skipping
  Running eval_run...
2026-05-01 16:08:42,517: Using the model weights from runs/total_precipitation_sum_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed555_3004_080530/model_epoch030.pt


/home/azureuser/miniconda3/envs/neuralhydrology/lib/python3.10/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.mod

...

# Evaluation: 100%|██████████| 127/127 [01:52<00:00,  1.13it/s]
2026-05-01 16:49:19,526: Stored metrics at runs/camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_3004_011932/test/model_epoch030/test_metrics.csv
2026-05-01 16:49:19,557: Stored results at runs/camels_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed777_3004_011932/test/model_epoch030/test_results.p
  Done!

Processing mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_031843...
  test_start_date already present, skipping
  test_end_date already present, skipping
  Running eval_run...
2026-05-01 16:49:19,570: Using the model weights from runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_031843/model_epoch030.pt


/home/azureuser/miniconda3/envs/neuralhydrology/lib/python3.10/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.mod

# Evaluation: 100%|██████████| 127/127 [01:53<00:00,  1.12it/s]
2026-05-01 16:51:12,977: Stored metrics at runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_031843/test/model_epoch030/test_metrics.csv
2026-05-01 16:51:13,007: Stored results at runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed666_0105_031843/test/model_epoch030/test_results.p
  Done!

Processing mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_235443...
  test_start_date already present, skipping
  test_end_date already present, skipping
  Running eval_run...
2026-05-01 16:51:13,019: Using the model weights from runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_235443/model_epoch030.pt


/home/azureuser/miniconda3/envs/neuralhydrology/lib/python3.10/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.mod

# Evaluation: 100%|██████████| 127/127 [01:53<00:00,  1.12it/s]
2026-05-01 16:53:06,097: Stored metrics at runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_235443/test/model_epoch030/test_metrics.csv
2026-05-01 16:53:06,127: Stored results at runs/mswep_precipitation_seq_270_1_epochs_30_hidden_256_dropout_04_fb_5_seed333_3004_235443/test/model_epoch030/test_results.p
  Done!

All runs completed!


In [13]:
# base_config_path = Path("basins_excluded.yml")

# with open(base_config_path, "r") as f:
#     base_config = yaml.safe_load(f)

In [7]:
# for r in range(1, len(precip_products) + 1):
#     for precip_combo in itertools.combinations(precip_products, r):
#         config = base_config.copy()

#         # Update dynamic inputs:
#         # Keep temperature and radiation, replace precip
#         config["dynamic_inputs"] = [
#             "surface_net_solar_radiation_mean",
#             "temperature_2m_max",
#             "temperature_2m_min",
#             *precip_combo
#         ]

#         # Create unique experiment name
#         precip_name = "_".join(precip_combo)
#         config["experiment_name"] = (
#             f"precip_{precip_name}"
#         )

#         # Save temporary config file
#         temp_config_path = Path(f"temp_{precip_name}")
#         with open(temp_config_path, "w") as f:
#             yaml.dump(config, f)

#         print(f"Running: {config['experiment_name']}")

#         # Run training
#         if torch.cuda.is_available() or torch.backends.mps.is_available():
#             start_run(config_file=temp_config_path)
#         else:
#             start_run(config_file=temp_config_path, gpu=-1)

In [1]:
# # by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
# if torch.cuda.is_available() or torch.backends.mps.is_available():
#     start_run(config_file=Path("basins_excluded.yml"))

# # fall back to CPU-only mode
# else:
#     start_run(config_file=Path("basins_excluded.yml"), gpu=-1)